# Modelo RENT: saldo mensual del propietario (Regresión Lineal)

**Objetivo:** predecir cuánto dinero le quedará al propietario de una vivienda al finalizar un mes
(`ingresos pagados - gastos pagados`), usando el comportamiento financiero de los meses anteriores.

Pasos:
1. Cargar el CSV de `financial_transactions`
2. Agregar por propiedad y por mes
3. Crear variables de meses anteriores (lags)
4. Entrenar `LinearRegression` con validación temporal
5. Evaluar contra una línea base
6. Exportar con `joblib`

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---- Configuración ----
USD_TO_COP = 4000.0     # tasa fija para unificar monedas (ajústala o usa la TRM real)
N_LAGS = 3              # cuántos meses anteriores mira el modelo
TEST_MONTHS = 3         # últimos meses reservados para evaluar
MODEL_PATH = "rent_model.joblib"

## 1. Cargar los datos

In [ ]:
try:
    from google.colab import files
    print("Sube el archivo financial_transactions.csv")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    CSV_PATH = "financial_transactions.csv"   # ejecución local

df = pd.read_csv(CSV_PATH)
for col in ["occurred_at", "due_date", "paid_at"]:
    df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)

print(df.shape)
df.head()

## 2. Agregación mensual por propiedad

- Solo cuenta como dinero real lo que está en estado `PAID`.
- La categoría `DEPOSIT` se excluye (el depósito de garantía no es ganancia del propietario).
- Los registros `PLANNED` (futuros) se ignoran.
- Todo se convierte a COP para que las propiedades en USD sean comparables.
- Los meses sin movimientos se rellenan con 0.

In [ ]:
df["amount_cop"] = np.where(df["currency"] == "USD", df["amount"] * USD_TO_COP, df["amount"])
df["month"] = df["occurred_at"].dt.to_period("M")

# El deposito de garantia no es ingreso del propietario (se devuelve), asi que no cuenta en el saldo
paid = df[(df["status"] == "PAID") & (df["category"] != "DEPOSIT")]
income = paid[paid["type"] == "INCOME"].groupby(["property_id", "month"])["amount_cop"].sum().rename("income")
expense = paid[paid["type"] == "EXPENSE"].groupby(["property_id", "month"])["amount_cop"].sum().rename("expense")
overdue = (
    df[(df["status"] == "OVERDUE") & (df["type"] == "INCOME")]
    .groupby(["property_id", "month"])["amount_cop"].sum().rename("overdue_income")
)

monthly = pd.concat([income, expense, overdue], axis=1).fillna(0.0)

# Rejilla completa propiedad x mes (rellena meses vacíos con 0)
# Los movimientos PLANNED (futuros) no definen el rango de meses
real = df[df["status"] != "PLANNED"]
all_months = pd.period_range(real["month"].min(), real["month"].max(), freq="M")
idx = pd.MultiIndex.from_product([df["property_id"].unique(), all_months], names=["property_id", "month"])
monthly = monthly.reindex(idx, fill_value=0.0).reset_index()

monthly["net_balance"] = monthly["income"] - monthly["expense"]   # <-- variable objetivo
monthly = monthly.sort_values(["property_id", "month"]).reset_index(drop=True)
monthly.head(10)

## 3. Variables (features)

Para predecir el mes *t* solo se usa información de meses **anteriores** (`shift`), así no hay fuga de datos:
saldo, ingresos y gastos de los últimos `N_LAGS` meses, ingreso en mora del mes previo,
promedio móvil de 3 meses, promedio histórico acumulado y estacionalidad (seno/coseno del mes).

In [ ]:
feature_cols = (
    [f"{c}_lag{l}" for l in range(1, N_LAGS + 1) for c in ["net_balance", "income", "expense"]]
    + ["overdue_income_lag1", "net_roll3", "net_hist_mean", "month_sin", "month_cos"]
)


def add_features(monthly):
    """Crea las variables a partir de meses ANTERIORES. Se reutiliza en entrenamiento e inferencia."""
    monthly = monthly.sort_values(["property_id", "month"]).reset_index(drop=True)
    g = monthly.groupby("property_id")

    for lag in range(1, N_LAGS + 1):
        for col in ["net_balance", "income", "expense"]:
            monthly[f"{col}_lag{lag}"] = g[col].shift(lag)

    monthly["overdue_income_lag1"] = g["overdue_income"].shift(1)
    monthly["net_roll3"] = g["net_balance"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    monthly["net_hist_mean"] = g["net_balance"].transform(lambda s: s.shift(1).expanding().mean())

    month_num = monthly["month"].dt.month
    monthly["month_sin"] = np.sin(2 * np.pi * month_num / 12)
    monthly["month_cos"] = np.cos(2 * np.pi * month_num / 12)
    return monthly


monthly = add_features(monthly)
data = monthly.dropna(subset=feature_cols).copy()
print("Filas de entrenamiento disponibles:", len(data))
print("Features:", len(feature_cols))

## 4. Entrenamiento con split temporal

In [ ]:
months_sorted = sorted(data["month"].unique())
cut = months_sorted[-TEST_MONTHS]

train = data[data["month"] < cut]
test = data[data["month"] >= cut]

X_train, y_train = train[feature_cols], train["net_balance"]
X_test, y_test = test[feature_cols], test["net_balance"]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("reg", LinearRegression()),
])
model.fit(X_train, y_train)

print(f"Train: {len(train)} filas | Test: {len(test)} filas (desde {cut})")

## 5. Evaluación

In [ ]:
def report(name, y_true, y_pred):
    err = np.asarray(y_true) - np.asarray(y_pred)
    mae = np.mean(np.abs(err))
    medae = np.median(np.abs(err))          # error "típico", poco sensible a meses atípicos
    rmse = np.sqrt(np.mean(err ** 2))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:<26} MAE: {mae:>10,.0f} | MedAE: {medae:>10,.0f} | RMSE: {rmse:>10,.0f} | R2: {r2:6.3f}")

pred = model.predict(X_test)

# Línea base ingenua: "el mes que viene será igual al anterior"
baseline = X_test["net_balance_lag1"]

report("Baseline (mes anterior)", y_test, baseline)
report("Regresión Lineal", y_test, pred)

### Validación más estable (rolling origin)

Con pocos meses de test, un solo mes atípico (vacancia o canon en mora) mueve mucho el R2.
Aquí se evalúa mes a mes sobre los últimos 6 meses: en cada paso se entrena solo con el pasado
y se predice el mes siguiente. Es la estimación más fiable de cómo se comportará en la app.

In [ ]:
from sklearn.base import clone

P, Y, B = [], [], []
for mth in months_sorted[-6:]:
    tr = data[data["month"] < mth]
    te = data[data["month"] == mth]
    m = clone(model).fit(tr[feature_cols], tr["net_balance"])
    P += list(m.predict(te[feature_cols]))
    Y += list(te["net_balance"])
    B += list(te["net_balance_lag1"])

report("Baseline (mes anterior)", Y, B)
report("Regresión Lineal", Y, P)

In [ ]:
coefs = pd.Series(model.named_steps["reg"].coef_, index=feature_cols).sort_values(key=abs, ascending=False)
print("Variables más influyentes (coeficientes estandarizados):")
coefs.head(8)

## 5.1 Gráficas de diagnóstico

Cómo leerlas:
- **Real vs predicho:** los puntos deben acercarse a la diagonal. Si forman una nube sin forma, el modelo no está aprendiendo.
- **Residuos:** el histograma debe estar centrado en 0 y sin sesgo; en el gráfico de residuos vs predicción no debe verse ningún patrón.
- **Serie temporal:** el saldo total real y el predicho deben moverse juntos mes a mes.
- **Coeficientes:** muestran qué variables pesan más en la predicción.
- **Train vs Test:** si el error en test es mucho mayor que en train, hay sobreajuste.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
M = 1e6  # mostrar en millones de COP

res = test[["property_id", "month", "net_balance"]].copy()
res["pred"] = pred
res["baseline"] = baseline.values
res["residual"] = res["net_balance"] - res["pred"]

# 1) Real vs predicho (modelo y baseline)
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
lim = [min(res["net_balance"].min(), res["pred"].min(), res["baseline"].min()) / M,
       max(res["net_balance"].max(), res["pred"].max(), res["baseline"].max()) / M]
for ax, col, title in zip(axes, ["pred", "baseline"], ["Regresión Lineal", "Baseline (mes anterior)"]):
    ax.scatter(res["net_balance"] / M, res[col] / M, alpha=0.5, s=18)
    ax.plot(lim, lim, "r--", label="Predicción perfecta")
    ax.set_title(f"{title}  |  R2 = {r2_score(res['net_balance'], res[col]):.3f}")
    ax.set_xlabel("Saldo real (millones COP)")
    ax.set_ylabel("Saldo predicho (millones COP)")
    ax.legend()
plt.suptitle("Real vs predicho (conjunto de test)")
plt.tight_layout()
plt.show()

In [ ]:
# 2) Residuos
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(res["residual"] / M, bins=30, edgecolor="white")
axes[0].axvline(0, color="r", linestyle="--")
axes[0].set_title(f"Distribución de residuos  (media = {res['residual'].mean() / M:.2f} M)")
axes[0].set_xlabel("Real - Predicho (millones COP)")

axes[1].scatter(res["pred"] / M, res["residual"] / M, alpha=0.5, s=18)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_title("Residuos vs predicción")
axes[1].set_xlabel("Saldo predicho (millones COP)")
axes[1].set_ylabel("Residuo (millones COP)")
plt.tight_layout()
plt.show()

In [ ]:
# 3) Serie temporal: saldo total por mes (real vs predicho vs baseline)
by_month = res.groupby("month")[["net_balance", "pred", "baseline"]].sum() / M
by_month.index = by_month.index.astype(str)

ax = by_month.plot(marker="o", figsize=(9, 4.5))
ax.set_title("Saldo total por mes en el periodo de test")
ax.set_ylabel("Millones COP")
ax.set_xlabel("Mes")
ax.legend(["Real", "Regresión Lineal", "Baseline"])
plt.tight_layout()
plt.show()

In [ ]:
# 4) Coeficientes del modelo
top = coefs.head(10).sort_values()
plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color=["tab:red" if v < 0 else "tab:blue" for v in top.values])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Variables más influyentes (coeficientes estandarizados)")
plt.xlabel("Coeficiente")
plt.tight_layout()
plt.show()

# 5) Train vs Test: ¿hay sobreajuste?
train_pred = model.predict(X_train)
summary = pd.DataFrame({
    "MAE": [mean_absolute_error(y_train, train_pred), mean_absolute_error(y_test, pred)],
    "R2": [r2_score(y_train, train_pred), r2_score(y_test, pred)],
}, index=["Train", "Test"])

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
(summary["MAE"] / M).plot.bar(ax=axes[0], color=["tab:green", "tab:orange"], rot=0)
axes[0].set_title("MAE (millones COP)")
summary["R2"].plot.bar(ax=axes[1], color=["tab:green", "tab:orange"], rot=0)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("R2")
plt.tight_layout()
plt.show()
summary

## 6. Exportar el modelo con joblib

In [ ]:
# Para producción se reentrena con TODOS los datos disponibles
from sklearn.base import clone
final_model = clone(model).fit(data[feature_cols], data["net_balance"])

artifact = {
    "model": final_model,
    "feature_cols": feature_cols,
    "n_lags": N_LAGS,
    "usd_to_cop": USD_TO_COP,
    "target": "net_balance (COP) = ingresos PAID - gastos PAID del mes",
}
joblib.dump(artifact, MODEL_PATH)
print("Modelo guardado en", MODEL_PATH)

try:
    from google.colab import files
    files.download(MODEL_PATH)
except ImportError:
    pass

## 7. Cargar el modelo y predecir el mes siguiente

`predict_next_month` recibe el historial mensual de las propiedades (`property_id`, `month`, `income`,
`expense`, `overdue_income`, `net_balance`) y devuelve el saldo esperado del mes que sigue al último.
En la app, el historial debe construirse igual que en la sección 2 (solo `PAID`, sin `DEPOSIT`, en COP)
y usando meses ya cerrados.

In [ ]:
loaded = joblib.load(MODEL_PATH)


def predict_next_month(history, art):
    nxt = history["month"].max() + 1
    future = pd.DataFrame({
        "property_id": history["property_id"].unique(), "month": nxt,
        "income": 0.0, "expense": 0.0, "overdue_income": 0.0, "net_balance": 0.0,   # se ignoran: solo se usan lags
    })
    full = add_features(pd.concat([history, future], ignore_index=True))
    rows = full[full["month"] == nxt].dropna(subset=art["feature_cols"]).copy()
    rows["pred_net_balance_cop"] = art["model"].predict(rows[art["feature_cols"]])
    return rows[["property_id", "month", "pred_net_balance_cop"]]


history = monthly[["property_id", "month", "income", "expense", "overdue_income", "net_balance"]]
next_pred = predict_next_month(history, loaded)
next_pred.head()